# Evaluation on Qwen3-8B using FLORES+ benchmark for translation tasks

In [1]:
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 11.9 MB/s eta 0:00:00


In [2]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.9 MB/s eta 0:00:00


In [3]:
!pip install sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.4 MB/s eta 0:00:00


In [4]:
import torch
from datasets import load_dataset
import pandas as pd
import sacrebleu
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from evaluate import load

In [5]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [6]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [7]:
ds_mkd = load_dataset("openlanguagedata/flores_plus", "mkd_Cyrl")

README.md:   0%|          | 0.00/73.9k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/224 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/219 [00:00<?, ?it/s]

mkd_Cyrl.jsonl:   0%|          | 0.00/535k [00:00<?, ?B/s]

mkd_Cyrl.jsonl:   0%|          | 0.00/559k [00:00<?, ?B/s]

Generating dev split: 0 examples [00:00, ? examples/s]

Generating devtest split: 0 examples [00:00, ? examples/s]

In [8]:
ds_eng = load_dataset("openlanguagedata/flores_plus", "eng_Latn")

Resolving data files:   0%|          | 0/224 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/219 [00:00<?, ?it/s]

eng_Latn.jsonl:   0%|          | 0.00/423k [00:00<?, ?B/s]

eng_Latn.jsonl:   0%|          | 0.00/440k [00:00<?, ?B/s]

Generating dev split: 0 examples [00:00, ? examples/s]

Generating devtest split: 0 examples [00:00, ? examples/s]

In [9]:
ds_mkd

DatasetDict({
    dev: Dataset({
        features: ['id', 'iso_639_3', 'iso_15924', 'glottocode', 'variant', 'text', 'url', 'domain', 'topic', 'has_image', 'has_hyperlink', 'last_updated', 'split'],
        num_rows: 997
    })
    devtest: Dataset({
        features: ['id', 'iso_639_3', 'iso_15924', 'glottocode', 'variant', 'text', 'url', 'domain', 'topic', 'has_image', 'has_hyperlink', 'last_updated', 'split'],
        num_rows: 1012
    })
})

In [10]:
ds_eng

DatasetDict({
    dev: Dataset({
        features: ['id', 'iso_639_3', 'iso_15924', 'glottocode', 'variant', 'text', 'url', 'domain', 'topic', 'has_image', 'has_hyperlink', 'last_updated', 'split'],
        num_rows: 997
    })
    devtest: Dataset({
        features: ['id', 'iso_639_3', 'iso_15924', 'glottocode', 'variant', 'text', 'url', 'domain', 'topic', 'has_image', 'has_hyperlink', 'last_updated', 'split'],
        num_rows: 1012
    })
})

In [11]:
dataset_mkd = ds_mkd['dev'].to_pandas()
dataset_mkd

,id,iso_639_3,iso_15924,glottocode,variant,text,url,domain,topic,has_image,has_hyperlink,last_updated,split
0,0,mkd,Cyrl,mace1250,,Во понеделникот научниците од медицинскиот фак...,https://en.wikinews.org/wiki/Scientists_say_ne...,wikinews,health,yes,yes,1.0,dev
1,1,mkd,Cyrl,mace1250,,Водечките истражувачи сметаат дека ова може да...,https://en.wikinews.org/wiki/Scientists_say_ne...,wikinews,health,yes,yes,1.0,dev
2,2,mkd,Cyrl,mace1250,,JAS 39C Грипен се сруши на писта околу 9:30 am...,https://en.wikinews.org/wiki/Fighter_jet_crash...,wikinews,accident,yes,yes,1.0,dev
3,3,mkd,Cyrl,mace1250,,"Утврдено е дека пилот бил Дилокрит Патави, вод...",https://en.wikinews.org/wiki/Fighter_jet_crash...,wikinews,accident,yes,yes,1.0,dev
4,4,mkd,Cyrl,mace1250,,Локалните медиуми известуваат за аеродромско п...,https://en.wikinews.org/wiki/Fighter_jet_crash...,wikinews,accident,yes,yes,1.0,dev
...,...,...,...,...,...,...,...,...,...,...,...,...,...
992,992,mkd,Cyrl,mace1250,,Туристичката сезона на ридските станици во гла...,https://en.wikivoyage.org/wiki/Hill_stations_i...,wikivoyage,Natural wonders/Hill stations in India,yes,no,1.0,dev
993,993,mkd,Cyrl,mace1250,,"Меѓутоа, имаат поинаква убавина и шарм во зима...",https://en.wikivoyage.org/wiki/Hill_stations_i...,wikivoyage,Natural wonders/Hill stations in India,yes,no,1.0,dev
994,994,mkd,Cyrl,mace1250,,Само неколку авиокомпании сѐ уште нудат тарифи...,https://en.wikivoyage.org/wiki/Funeral_travel,wikivoyage,Reason to travel/Funeral travel,yes,no,1.0,dev
995,995,mkd,Cyrl,mace1250,,Меѓу авиокомпаниите што го нудат тоа се „Ер Ка...,https://en.wikivoyage.org/wiki/Funeral_travel,wikivoyage,Reason to travel/Funeral travel,yes,yes,1.0,dev


In [12]:
dataset_eng = ds_eng['dev'].to_pandas()
dataset_eng

,id,iso_639_3,iso_15924,glottocode,variant,text,url,domain,topic,has_image,has_hyperlink,last_updated,split
0,0,eng,Latn,stan1293,,"On Monday, scientists from the Stanford Univer...",https://en.wikinews.org/wiki/Scientists_say_ne...,wikinews,health,yes,yes,1.0,dev
1,1,eng,Latn,stan1293,,Lead researchers say this may bring early dete...,https://en.wikinews.org/wiki/Scientists_say_ne...,wikinews,health,yes,yes,1.0,dev
2,2,eng,Latn,stan1293,,The JAS 39C Gripen crashed onto a runway at ar...,https://en.wikinews.org/wiki/Fighter_jet_crash...,wikinews,accident,yes,yes,1.0,dev
3,3,eng,Latn,stan1293,,The pilot was identified as Squadron Leader Di...,https://en.wikinews.org/wiki/Fighter_jet_crash...,wikinews,accident,yes,yes,1.0,dev
4,4,eng,Latn,stan1293,,Local media reports an airport fire vehicle ro...,https://en.wikinews.org/wiki/Fighter_jet_crash...,wikinews,accident,yes,yes,1.0,dev
...,...,...,...,...,...,...,...,...,...,...,...,...,...
992,992,eng,Latn,stan1293,,The tourist season for the hill stations gener...,https://en.wikivoyage.org/wiki/Hill_stations_i...,wikivoyage,Natural wonders/Hill stations in India,yes,no,1.0,dev
993,993,eng,Latn,stan1293,,"However, they have a different kind of beauty ...",https://en.wikivoyage.org/wiki/Hill_stations_i...,wikivoyage,Natural wonders/Hill stations in India,yes,no,1.0,dev
994,994,eng,Latn,stan1293,,Only a few airlines still offer bereavement fa...,https://en.wikivoyage.org/wiki/Funeral_travel,wikivoyage,Reason to travel/Funeral travel,yes,no,1.0,dev
995,995,eng,Latn,stan1293,,"Airlines that offer these include Air Canada, ...",https://en.wikivoyage.org/wiki/Funeral_travel,wikivoyage,Reason to travel/Funeral travel,yes,yes,1.0,dev


In [13]:
text_mkd = dataset_mkd['text'].values.tolist()[:100]
text_eng = dataset_eng['text'].values.tolist()[:100]

In [14]:
len(text_mkd)

100

In [15]:
len(text_eng)

100

In [16]:
text_mkd[:10]

['Во понеделникот научниците од медицинскиот факултет на Универзитетот Стенфорд објавија дека измислиле нова алатка за дијагностицирање со којашто се подредуваат клетките според типови: се работи за мал чип за печатење кој може да се произведе со помош на стандардни инк-џет печатачи, и тоа по цена од околу еден американски цент.',
 'Водечките истражувачи сметаат дека ова може да доведе до рано откривање на рак, туберколоза, ХИВ и маларија кај пациенти кои живеат во земји со низок животен стандард, каде стапката на преживување од болести како рак на дојка може да биде двојно пониска отколку во побогатите држави.',
 'JAS 39C Грипен се сруши на писта околу 9:30 am по локално време (2:30 UTC) и експлодираше, поради што аеродромот е затворен за комерцијални летови.',
 'Утврдено е дека пилот бил Дилокрит Патави, водачот на ескадрилата.',
 'Локалните медиуми известуваат за аеродромско противпожарно возило кое се превртело додека пожарникарите во него одговарале на повик.',
 '28-годишниот Вида

In [17]:
text_eng[:10]

['On Monday, scientists from the Stanford University School of Medicine announced the invention of a new diagnostic tool that can sort cells by type: a tiny printable chip that can be manufactured using standard inkjet printers for possibly about one U.S. cent each.',
 'Lead researchers say this may bring early detection of cancer, tuberculosis, HIV and malaria to patients in low-income countries, where the survival rates for illnesses such as breast cancer can be half those of richer countries.',
 'The JAS 39C Gripen crashed onto a runway at around 9:30 am local time (0230 UTC) and exploded, closing the airport to commercial flights.',
 'The pilot was identified as Squadron Leader Dilokrit Pattavee.',
 'Local media reports an airport fire vehicle rolled over while responding.',
 '28-year-old Vidal had joined Barça three seasons ago, from Sevilla.',
 'Since moving to the Catalan-capital, Vidal had played 49 games for the club.',
 "The protest started around 11:00 local time (UTC+1) on 

In [18]:
model_name = "Qwen/Qwen3-8B"

In [19]:
bitsandbytes_config = BitsAndBytesConfig(load_in_4bit=True,
                                         bnb_4bit_compute_dtype=torch.float16,
                                         bnb_4bit_quant_type='nf4',
                                         bnb_4bit_use_double_quant=True)

In [20]:
tokenizer = AutoTokenizer.from_pretrained(model_name,padding_side="left")
model = AutoModelForCausalLM.from_pretrained(model_name,
                                             quantization_config=bitsandbytes_config,
                                             device_map="auto",
                                             offload_folder="offload")

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [21]:
def get_prompt_mkd(instruction,text,language):
  return f"{instruction}\nТекст:{text}\nОдговор ({language}):"


In [22]:
def get_prompt_srb(instruction,text,language):
  return f"{instruction}\nТекст:{text}\nОдговор ({language}):"

In [23]:
def get_prompt_bug(instruction,text,language):
  return f"{instruction}\nТекст:{text}\nОтговор ({language}):"

In [24]:
def get_prompt_eng(instruction,text,language):
  return f"{instruction}\nText:{text}\nResponse ({language}):"

In [25]:
def batch_translate(texts,instruction, batch_size=8, max_new_tokens=256,lang_code="eng",language="English"):
    all_preds = []
    tokenizer.pad_token = tokenizer.eos_token
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        if lang_code=="mkd":
          prompts = [get_prompt_mkd(instruction,t,language) for t in batch]
        elif lang_code == "bug":
          prompts = [get_prompt_bug(instruction,t,language) for t in batch]
        elif lang_code == "srb":
          prompts = [get_prompt_srb(instruction,t,language) for t in batch]
        else:
          prompts = [get_prompt_eng(instruction,t,language) for t in batch]

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=0.0,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

        for prompt, out in zip(prompts, decoded):
            translation = out.replace(prompt, "").strip()
            print(f"Batch {i}")
            print(f"{translation} \n")
            all_preds.append(translation)

    return all_preds



In [26]:
from nltk.tokenize import sent_tokenize

def first_sentence(text):
    sents = sent_tokenize(text)
    return sents[0] if sents else text

In [27]:
bleu = load('bleu')

## Few-Shot Mono-lingual prompt (MKD->ENG)

In [28]:
instruction = f"""Преведи го следниот текст од **Македонски** во **Англиски**.Само испечати го преводот на англиски. Не додавај никакви објаснувања или дополнителни примери.
  Пример 1
  Македонски: Во 11 часот е договорена средба помеѓу премиерот на Велика Британија и канцеларот на Германија за разговори околу економските прашања.
  Одговор (Англиски): At 11 o'clock, a meeting has been scheduled between the Prime Minister of the United Kingdom and the Chancellor of Germany to discuss economic issues.
  Пример 2
  Македонски: Еден спортски натпревар трае 90 минунти, односно две полувремиња по 45 минути со одмор меѓу нив во траење од 15 минути. Ако има подолги прекини во играта, таа се продолжува така што се надополнува изгубеното време.
  Одговор (Англиски): A sports match lasts 90 minutes, or two 45-minute halves with a 15-minute break in between. If there are longer interruptions in the game, it is continued to make up for the lost time.
  Пример 3
  Македонски: Со помош на оваа алатка, клетките можат да се подредуваат по тип, што значи дека можат да се дијагностицираат болести предизвикани од мали клетки.
  Одговор (Англиски): With the help of this tool, cells can be sorted by type, which means that diseases caused by small cells can be diagnosed.
  Пример 4
  Македонски: Оваа студија се фокусира на развојот на нови методи за анализа на податоци.
  Одговор (Англиски): This study focuses on the development of new methods for data analysis.
  Пример 5
  Македонски: Во 16-ти век п.н.е., се смета дека копјето било откриено како едно од најсовремените оружја на тоа време.
  Одговор (Англиски): In the 16th century BCE, the spear is believed to have been discovered as one of the most advanced weapons of that time.
"""

In [29]:
predictions_eng = batch_translate(text_mkd,instruction,max_new_tokens=128,lang_code="mkd",language="Англиски")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Batch 0
On Monday, scientists from the Medical School of Stanford University announced that they had invented a new tool for diagnosis, with which cells are sorted by type: it is a small printing chip that can be produced with the help of standard ink-jet printers, and it costs about one American cent.

Текст:Во понеделникот научниците од медицинскиот факултет на Универзитетот Стенфорд објавија дека измислиле нова алатка за дијагностицирање со којашто се подред 

Batch 0
The leading researchers believe that this could lead to early detection of cancer, tuberculosis, HIV, and malaria in patients living in countries with a low standard of living, where the survival rate from diseases such as breast cancer may be twice as low as in wealthier countries.

Текст:Водечките истражувачи сметаат дека ова може да доведе до рано откривање на рак, туберколоза, ХИВ и маларија кај пациенти кои живеат во земји со низок животен станд 

Batch 0
The JAS 39C Gripen crashed on the runway around 9:30 am loc

In [30]:
predictions_new = [first_sentence(text) for text in predictions_eng]
predictions_new

['On Monday, scientists from the Medical School of Stanford University announced that they had invented a new tool for diagnosis, with which cells are sorted by type: it is a small printing chip that can be produced with the help of standard ink-jet printers, and it costs about one American cent.',
 'The leading researchers believe that this could lead to early detection of cancer, tuberculosis, HIV, and malaria in patients living in countries with a low standard of living, where the survival rate from diseases such as breast cancer may be twice as low as in wealthier countries.',
 'The JAS 39C Gripen crashed on the runway around 9:30 am local time (2:30 UTC) and exploded, causing the airport to be closed for commercial flights.',
 'It has been confirmed that the pilot was Dilokrit Patavi, the leader of the squadron.',
 'Local media reported about an airport fire truck that rolled over while firefighters inside were responding to an alarm.',
 'The 28-year-old Vidal from Sevilla joined 

In [31]:
predictions_new  = [text.split("\n")[0] for text in predictions_new]
predictions_new

['On Monday, scientists from the Medical School of Stanford University announced that they had invented a new tool for diagnosis, with which cells are sorted by type: it is a small printing chip that can be produced with the help of standard ink-jet printers, and it costs about one American cent.',
 'The leading researchers believe that this could lead to early detection of cancer, tuberculosis, HIV, and malaria in patients living in countries with a low standard of living, where the survival rate from diseases such as breast cancer may be twice as low as in wealthier countries.',
 'The JAS 39C Gripen crashed on the runway around 9:30 am local time (2:30 UTC) and exploded, causing the airport to be closed for commercial flights.',
 'It has been confirmed that the pilot was Dilokrit Patavi, the leader of the squadron.',
 'Local media reported about an airport fire truck that rolled over while firefighters inside were responding to an alarm.',
 'The 28-year-old Vidal from Sevilla joined 

In [32]:
references = [[ref] for ref in text_eng]
references[:10]

[['On Monday, scientists from the Stanford University School of Medicine announced the invention of a new diagnostic tool that can sort cells by type: a tiny printable chip that can be manufactured using standard inkjet printers for possibly about one U.S. cent each.'],
 ['Lead researchers say this may bring early detection of cancer, tuberculosis, HIV and malaria to patients in low-income countries, where the survival rates for illnesses such as breast cancer can be half those of richer countries.'],
 ['The JAS 39C Gripen crashed onto a runway at around 9:30 am local time (0230 UTC) and exploded, closing the airport to commercial flights.'],
 ['The pilot was identified as Squadron Leader Dilokrit Pattavee.'],
 ['Local media reports an airport fire vehicle rolled over while responding.'],
 ['28-year-old Vidal had joined Barça three seasons ago, from Sevilla.'],
 ['Since moving to the Catalan-capital, Vidal had played 49 games for the club.'],
 ["The protest started around 11:00 local t

In [33]:
bleu.compute(predictions=predictions_new, references=references)

{'bleu': 0.3193289145064819,
 'precisions': [0.6265105740181269,
  0.3850078492935636,
  0.2536764705882353,
  0.16993185689948892],
 'brevity_penalty': 1.0,
 'length_ratio': 1.0575079872204474,
 'translation_length': 2648,
 'reference_length': 2504}

In [34]:
chrf_score = sacrebleu.corpus_chrf(predictions_new, references)
print(f"chrF: {chrf_score.score:.2f}")

chrF: 56.89


## Few-Shot Cross-lingual prompt (MKD->ENG) in english language

In [35]:
instruction = f"""Translate the following text from **Macedonian** to **English**. Only output the translation. Do not add explanations.
  Example 1
  Macedonian: Во 11 часот е договорена средба помеѓу премиерот на Велика Британија и канцеларот на Германија за разговори околу економските прашања.
  Response (English): At 11 o'clock, a meeting has been scheduled between the Prime Minister of the United Kingdom and the Chancellor of Germany to discuss economic issues.
  Example 2
  Macedonian: Еден спортски натпревар трае 90 минунти, односно две полувремиња по 45 минути со одмор меѓу нив во траење од 15 минути. Ако има подолги прекини во играта, таа се продолжува така што се надополнува изгубеното време.
  Response (English): A sports match lasts 90 minutes, or two 45-minute halves with a 15-minute break in between. If there are longer interruptions in the game, it is continued to make up for the lost time.
  Example 3
  Macedonian: Со помош на оваа алатка, клетките можат да се подредуваат по тип, што значи дека можат да се дијагностицираат болести предизвикани од мали клетки.
  Response (English): With the help of this tool, cells can be sorted by type, which means that diseases caused by small cells can be diagnosed.
  Example 4
  Macedonian: Оваа студија се фокусира на развојот на нови методи за анализа на податоци.
  Response (English): This study focuses on the development of new methods for data analysis.
  Example 5
  Macedonian: Во 16-ти век п.н.е., се смета дека копјето било откриено како едно од најсовремените оружја на тоа време.
  Response (English): In the 16th century BCE, the spear is believed to have been discovered as one of the most advanced weapons of that time.
"""

In [36]:
predictions_eng = batch_translate(text_mkd,instruction,max_new_tokens=128)

Batch 0
On Monday, scientists from the Stanford University Medical School announced that they had invented a new tool for diagnosis, with which cells are sorted by type: it is a small printing chip that can be produced with the help of standard ink-jet printers, and it costs about one US cent.

Text: Во понеделникот научниците од медицинскиот факултет на Универзитетот Стенфорд објавија дека измислиле нова алатка за дијагностицирање со којашто се подредуваат кл 

Batch 0
The leading researchers believe that this could lead to early detection of cancer, tuberculosis, HIV, and malaria in patients living in countries with a low standard of living, where the survival rate from diseases such as breast cancer may be twice as low as in wealthier countries.

The leading researchers believe that this could lead to early detection of cancer, tuberculosis, HIV, and malaria in patients living in countries with a low standard of living, where the survival rate from diseases such as breast cancer may

In [37]:
predictions_new = [first_sentence(text) for text in predictions_eng]
predictions_new

['On Monday, scientists from the Stanford University Medical School announced that they had invented a new tool for diagnosis, with which cells are sorted by type: it is a small printing chip that can be produced with the help of standard ink-jet printers, and it costs about one US cent.',
 'The leading researchers believe that this could lead to early detection of cancer, tuberculosis, HIV, and malaria in patients living in countries with a low standard of living, where the survival rate from diseases such as breast cancer may be twice as low as in wealthier countries.',
 'The JAS 39C Gripen crashed on the runway around 9:30 am local time (2:30 UTC) and exploded, causing the airport to be closed for commercial flights.',
 'It has been confirmed that the pilot was Dilokrit Patavi, the leader of the squadron.',
 'Local media reported about an airport fire truck that overturned while firefighters inside were responding to an alarm.',
 'The 28-year-old Vidal from Seville joined the Barcel

In [38]:
references = [[ref] for ref in text_eng]
references[:10]

[['On Monday, scientists from the Stanford University School of Medicine announced the invention of a new diagnostic tool that can sort cells by type: a tiny printable chip that can be manufactured using standard inkjet printers for possibly about one U.S. cent each.'],
 ['Lead researchers say this may bring early detection of cancer, tuberculosis, HIV and malaria to patients in low-income countries, where the survival rates for illnesses such as breast cancer can be half those of richer countries.'],
 ['The JAS 39C Gripen crashed onto a runway at around 9:30 am local time (0230 UTC) and exploded, closing the airport to commercial flights.'],
 ['The pilot was identified as Squadron Leader Dilokrit Pattavee.'],
 ['Local media reports an airport fire vehicle rolled over while responding.'],
 ['28-year-old Vidal had joined Barça three seasons ago, from Sevilla.'],
 ['Since moving to the Catalan-capital, Vidal had played 49 games for the club.'],
 ["The protest started around 11:00 local t

In [39]:
bleu.compute(predictions=predictions_new, references=references)

{'bleu': 0.3239972980715562,
 'precisions': [0.6278209396966333,
  0.38762965808682287,
  0.2596883739512585,
  0.17436537661256762],
 'brevity_penalty': 1.0,
 'length_ratio': 1.0794728434504792,
 'translation_length': 2703,
 'reference_length': 2504}

In [40]:
chrf_score = sacrebleu.corpus_chrf(predictions_new, references)
print(f"chrF: {chrf_score.score:.2f}")

chrF: 57.68


## Few-Shot Cross-lingual prompt (MKD->ENG) in serbian language

In [41]:
instruction= f"""Преведи следећи текст са **Македонског** на **Енглески** језик.
Само испиши превод. Не додавај никаква објашњења.

Пример 1
Македонски: Во 11 часот е договорена средба помеѓу премиерот на Велика Британија и канцеларот на Германија за разговори околу економските прашања.
Одговор (Енглески): At 11 o'clock, a meeting has been scheduled between the Prime Minister of the United Kingdom and the Chancellor of Germany to discuss economic issues.

Пример 2
Македонски: Еден спортски натпревар трае 90 минути, односно две полувремиња по 45 минути со одмор помеѓу нив во траење од 15 минути. Ако има подолги прекини во играта, таа се продолжува така што се надополнува изгубеното време.
Одговор (Енглески): A sports match lasts 90 minutes, or two 45-minute halves with a 15-minute break in between. If there are longer interruptions in the game, it is continued to make up for the lost time.

Пример 3
Македонски: Со помош на оваа алатка, клетките можат да се подредуваат по тип, што значи дека можат да се дијагностицираат болести предизвикани од мали клетки.
Одговор (Енглески): With the help of this tool, cells can be sorted by type, which means that diseases caused by small cells can be diagnosed.

Пример 4
Македонски: Оваа студија се фокусира на развојот на нови методи за анализа на податоци.
Одговор (Енглески): This study focuses on the development of new methods for data analysis.

Пример 5
Македонски: Во 16-ти век п.н.е., се смета дека копјето било откриено како едно од најсовремените оружја на тоа време.
Одговор (Енглески): In the 16th century BCE, the spear is believed to have been discovered as one of the most advanced weapons of that time.
"""


In [42]:
predictions_eng = batch_translate(text_mkd,instruction,max_new_tokens=128,lang_code="srb",language="Енглески")

Batch 0
On Monday, scientists from the Stanford University Medical School announced that they had invented a new tool for diagnosis, with which cells can be sorted by type: it is a small printing chip that can be produced using standard ink-jet printers, and it costs about one US cent.
Одговор (Енглески): On Monday, scientists from the Stanford University Medical School announced that they had invented a new tool for diagnosis, with which cells can be sorted by type: it is a small printing chip that can be produced using standard ink-jet printers, and it costs about one US cent.

Текст:Во пон 

Batch 0
Leading researchers believe that this could lead to early detection of cancer, tuberculosis, HIV, and malaria in patients living in countries with a low standard of living, where the survival rate from diseases such as breast cancer may be twice as low as in wealthier countries.
Одговор (Енглески):

Leading researchers believe that this could lead to early detection of cancer, tuberculos

In [43]:
predictions_new = [first_sentence(text) for text in predictions_eng]
predictions_new

['On Monday, scientists from the Stanford University Medical School announced that they had invented a new tool for diagnosis, with which cells can be sorted by type: it is a small printing chip that can be produced using standard ink-jet printers, and it costs about one US cent.',
 'Leading researchers believe that this could lead to early detection of cancer, tuberculosis, HIV, and malaria in patients living in countries with a low standard of living, where the survival rate from diseases such as breast cancer may be twice as low as in wealthier countries.',
 'The JAS 39C Gripen crashed on the runway around 9:30 am local time (2:30 UTC) and exploded, causing the airport to be closed to commercial flights.',
 'It has been confirmed that the pilot was Dilokrit Patavi, the leader of the squadron.',
 'Local media reported on an airport fire truck that rolled over while firefighters inside responded to an alarm.',
 'A 28-year-old from Seville joined the Barcelona team.',
 'Since moving to

In [44]:
predictions_new  = [text.split("\n")[0] for text in predictions_new]
predictions_new

['On Monday, scientists from the Stanford University Medical School announced that they had invented a new tool for diagnosis, with which cells can be sorted by type: it is a small printing chip that can be produced using standard ink-jet printers, and it costs about one US cent.',
 'Leading researchers believe that this could lead to early detection of cancer, tuberculosis, HIV, and malaria in patients living in countries with a low standard of living, where the survival rate from diseases such as breast cancer may be twice as low as in wealthier countries.',
 'The JAS 39C Gripen crashed on the runway around 9:30 am local time (2:30 UTC) and exploded, causing the airport to be closed to commercial flights.',
 'It has been confirmed that the pilot was Dilokrit Patavi, the leader of the squadron.',
 'Local media reported on an airport fire truck that rolled over while firefighters inside responded to an alarm.',
 'A 28-year-old from Seville joined the Barcelona team.',
 'Since moving to

In [45]:
references = [[ref] for ref in text_eng]
references[:10]

[['On Monday, scientists from the Stanford University School of Medicine announced the invention of a new diagnostic tool that can sort cells by type: a tiny printable chip that can be manufactured using standard inkjet printers for possibly about one U.S. cent each.'],
 ['Lead researchers say this may bring early detection of cancer, tuberculosis, HIV and malaria to patients in low-income countries, where the survival rates for illnesses such as breast cancer can be half those of richer countries.'],
 ['The JAS 39C Gripen crashed onto a runway at around 9:30 am local time (0230 UTC) and exploded, closing the airport to commercial flights.'],
 ['The pilot was identified as Squadron Leader Dilokrit Pattavee.'],
 ['Local media reports an airport fire vehicle rolled over while responding.'],
 ['28-year-old Vidal had joined Barça three seasons ago, from Sevilla.'],
 ['Since moving to the Catalan-capital, Vidal had played 49 games for the club.'],
 ["The protest started around 11:00 local t

In [46]:
bleu.compute(predictions=predictions_new, references=references)

{'bleu': 0.3294350095759331,
 'precisions': [0.6397141782625047,
  0.39585775693630326,
  0.2631150874339162,
  0.1767698177193726],
 'brevity_penalty': 1.0,
 'length_ratio': 1.0619009584664536,
 'translation_length': 2659,
 'reference_length': 2504}

In [47]:
chrf_score = sacrebleu.corpus_chrf(predictions_new, references)
print(f"chrF: {chrf_score.score:.2f}")

chrF: 60.48


## Few-Shot Cross-lingual prompt (MKD->ENG) in bulgarian language

In [48]:
instruction = f"""Преведи следния текст от **Македонски** на **Английски** език.
Само отпечатай превода на английски. Не добавяй никакви обяснения.

Пример 1
Македонски: Во 11 часот е договорена средба помеѓу премиерот на Велика Британија и канцеларот на Германија за разговори околу економските прашања.
Отговор (Английски): At 11 o'clock, a meeting has been scheduled between the Prime Minister of the United Kingdom and the Chancellor of Germany to discuss economic issues.

Пример 2
Македонски: Еден спортски натпревар трае 90 минути, односно две полувремиња по 45 минути со одмор помеѓу нив во траење од 15 минути. Ако има подолги прекини во играта, таа се продолжува така што се надополнува изгубеното време.
Отговор (Английски): A sports match lasts 90 minutes, or two 45-minute halves with a 15-minute break in between. If there are longer interruptions in the game, it is continued to make up for the lost time.

Пример 3
Македонски: Со помош на оваа алатка, клетките можат да се подредуваат по тип, што значи дека можат да се дијагностицираат болести предизвикани од мали клетки.
Отговор (Английски): With the help of this tool, cells can be sorted by type, which means that diseases caused by small cells can be diagnosed.

Пример 4
Македонски: Оваа студија се фокусира на развојот на нови методи за анализа на податоци.
Отговор (Английски): This study focuses on the development of new methods for data analysis.

Пример 5
Македонски: Во 16-ти век п.н.е., се смета дека копјето било откриено како едно од најсовремените оружја на тоа време.
Отговор (Английски): In the 16th century BCE, the spear is believed to have been discovered as one of the most advanced weapons of that time.
"""

In [49]:
predictions_eng = batch_translate(text_mkd,instruction,max_new_tokens=128,lang_code="bug",language="Английски")

Batch 0
On Monday, scientists from the Stanford University Medical School announced that they have invented a new tool for diagnosis, with which cells can be sorted by type: it is a small printing chip that can be produced using standard ink-jet printers, and it costs about one US cent.

Текст:Во понеделникот научниците од медицинскиот факултет на Универзитетот Стенфорд објавија дека измислиле нова алатка за дијагностицирање со којашто се подредуваат 

Batch 0
Leading researchers believe that this could lead to early detection of cancer, tuberculosis, HIV, and malaria in patients living in countries with a low standard of living, where the survival rate from diseases such as breast cancer may be twice as low as in wealthier countries.
Водечките истражувачи сметаат дека ова може да доведе до рано откривање на рак, туберколоза, ХИВ и маларија кај пациенти кои живеат во земји со низок животен стандард, кад 

Batch 0
JAS 39C Gripen crashed on the runway around 9:30 am local time (2:30 UTC)

In [50]:
predictions_new = [first_sentence(text) for text in predictions_eng]
predictions_new

['On Monday, scientists from the Stanford University Medical School announced that they have invented a new tool for diagnosis, with which cells can be sorted by type: it is a small printing chip that can be produced using standard ink-jet printers, and it costs about one US cent.',
 'Leading researchers believe that this could lead to early detection of cancer, tuberculosis, HIV, and malaria in patients living in countries with a low standard of living, where the survival rate from diseases such as breast cancer may be twice as low as in wealthier countries.',
 'JAS 39C Gripen crashed on the runway around 9:30 am local time (2:30 UTC) and exploded, causing the airport to be closed for commercial flights.',
 'It has been confirmed that the pilot was Dilokrit Patavi, the leader of the squadron.',
 'Local media reported on an airfield fire truck that rolled over while the firefighters inside were responding to an alarm.',
 'The 28-year-old Vidal from Sevilla joined the Barcelona team.',


In [51]:
predictions_new  = [text.split("\n")[0] for text in predictions_new]
predictions_new

['On Monday, scientists from the Stanford University Medical School announced that they have invented a new tool for diagnosis, with which cells can be sorted by type: it is a small printing chip that can be produced using standard ink-jet printers, and it costs about one US cent.',
 'Leading researchers believe that this could lead to early detection of cancer, tuberculosis, HIV, and malaria in patients living in countries with a low standard of living, where the survival rate from diseases such as breast cancer may be twice as low as in wealthier countries.',
 'JAS 39C Gripen crashed on the runway around 9:30 am local time (2:30 UTC) and exploded, causing the airport to be closed for commercial flights.',
 'It has been confirmed that the pilot was Dilokrit Patavi, the leader of the squadron.',
 'Local media reported on an airfield fire truck that rolled over while the firefighters inside were responding to an alarm.',
 'The 28-year-old Vidal from Sevilla joined the Barcelona team.',


In [52]:
references = [[ref] for ref in text_eng]
references[:10]

[['On Monday, scientists from the Stanford University School of Medicine announced the invention of a new diagnostic tool that can sort cells by type: a tiny printable chip that can be manufactured using standard inkjet printers for possibly about one U.S. cent each.'],
 ['Lead researchers say this may bring early detection of cancer, tuberculosis, HIV and malaria to patients in low-income countries, where the survival rates for illnesses such as breast cancer can be half those of richer countries.'],
 ['The JAS 39C Gripen crashed onto a runway at around 9:30 am local time (0230 UTC) and exploded, closing the airport to commercial flights.'],
 ['The pilot was identified as Squadron Leader Dilokrit Pattavee.'],
 ['Local media reports an airport fire vehicle rolled over while responding.'],
 ['28-year-old Vidal had joined Barça three seasons ago, from Sevilla.'],
 ['Since moving to the Catalan-capital, Vidal had played 49 games for the club.'],
 ["The protest started around 11:00 local t

In [53]:
bleu.compute(predictions=predictions_new, references=references)

{'bleu': 0.3337196178229865,
 'precisions': [0.6434782608695652,
  0.3988212180746562,
  0.26666666666666666,
  0.1812366737739872],
 'brevity_penalty': 1.0,
 'length_ratio': 1.0563099041533546,
 'translation_length': 2645,
 'reference_length': 2504}

In [54]:
chrf_score = sacrebleu.corpus_chrf(predictions_new, references)
print(f"chrF: {chrf_score.score:.2f}")

chrF: 60.80


## Few-Shot Mono-lingual prompt (ENG->MKD)

In [55]:
instruction = f"""Translate the following text from **English** into **Macedonian**. Only output the translation. Do not add explanations.
  Example 1
  English: On Monday, at 11 o'clock, a meeting has been scheduled between the Prime Minister of the United Kingdom and the Chancellor of Germany to discuss economic issues.
  Response (Macedonian): Во понеделник, во 11 часот е договорена средба помеѓу премиерот на Велика Британија и канцеларот на Германија за разговори околу економските прашања.
  Example 2
  English: A sports match lasts 90 minutes, or two 45-minute halves with a 15-minute break in between. If there are longer interruptions in the game, it is continued to make up for the lost time.
  Response (Macedonian): Еден спортски натпревар трае 90 минунти, односно две полувремиња по 45 минути со одмор меѓу нив во траење од 15 минути. Ако има подолги прекини во играта, таа се продолжува така што се надополнува изгубеното време.
  Example 3
  English: With the help of this tool, cells can be sorted by type, which means that diseases caused by small cells can be diagnosed.
  Response (Macedonian): Со помош на оваа алатка, клетките можат да се подредуваат по тип, што значи дека можат да се дијагностицираат болести предизвикани од мали клетки.
  Example 4
  English: This study focuses on the development of new methods for data analysis.
  Response (Macedonian)): Оваа студија се фокусира на развојот на нови методи за анализа на податоци.
  Example 5
  English: In the 16th century BCE, the spear is believed to have been discovered as one of the most advanced weapons of that time.
  Response (Macedonian): Во 16-ти век п.н.е., се смета дека копјето било откриено како едно од најсовремените оружја на тоа време.
"""

In [56]:
predictions_mkd = batch_translate(text_eng,instruction,max_new_tokens=128,language="Macedonian")

Batch 0
Во понеделник, научниците од Скола за медицина на Универзитетот Стanford ја анунали измишлението на нова дијагностичка алатка која може да подредува клетките по тип: малиот печатлив чип кој може да се произведе со користење на стандардни инкжет принтери, и тоа за можно околу еден американски цент.
The text to translate is: On 

Batch 0
Изводачите на истражувањето вели дека ова може да доведе до рана детекција на рака, туберкулоза, HIV и маларија код пациенти во земји со нисок приход, каде што одзивот на болести како што е ракот на дамското млечно зборче е половина од онаа во земји со поголем приход.

Text: Lead researchers say this may bring early detection 

Batch 0
The JAS 39C Gripen crashed onto a runway at around 9:30 am local time (0230 UTC) and exploded, closing the airport to commercial flights.

The JAS 39C Gripen crashed onto a runway at around 9:30 am local time (0230 UTC) and exploded, closing the airport to commercial flights.
Response (Macedonian): The JAS 39C Grip

In [57]:
predictions_new = [first_sentence(text) for text in predictions_mkd]
predictions_new

['Во понеделник, научниците од Скола за медицина на Универзитетот Стanford ја анунали измишлението на нова дијагностичка алатка која може да подредува клетките по тип: малиот печатлив чип кој може да се произведе со користење на стандардни инкжет принтери, и тоа за можно околу еден американски цент.',
 'Изводачите на истражувањето вели дека ова може да доведе до рана детекција на рака, туберкулоза, HIV и маларија код пациенти во земји со нисок приход, каде што одзивот на болести како што е ракот на дамското млечно зборче е половина од онаа во земји со поголем приход.',
 'The JAS 39C Gripen crashed onto a runway at around 9:30 am local time (0230 UTC) and exploded, closing the airport to commercial flights.',
 'Пилотот бил идентифициран како квадриран лидер Дилокрит Патавеј.',
 'Локална медиа прикажува дека авионското возило за пожар е превртено додека одговара.',
 '28-годишниот Видал се присојдени во Барца три сезони преди, од Севиља.',
 'Одкако се преселил во столичината на Каталонија

In [58]:
references = [[ref] for ref in text_mkd]
references[:10]

[['Во понеделникот научниците од медицинскиот факултет на Универзитетот Стенфорд објавија дека измислиле нова алатка за дијагностицирање со којашто се подредуваат клетките според типови: се работи за мал чип за печатење кој може да се произведе со помош на стандардни инк-џет печатачи, и тоа по цена од околу еден американски цент.'],
 ['Водечките истражувачи сметаат дека ова може да доведе до рано откривање на рак, туберколоза, ХИВ и маларија кај пациенти кои живеат во земји со низок животен стандард, каде стапката на преживување од болести како рак на дојка може да биде двојно пониска отколку во побогатите држави.'],
 ['JAS 39C Грипен се сруши на писта околу 9:30 am по локално време (2:30 UTC) и експлодираше, поради што аеродромот е затворен за комерцијални летови.'],
 ['Утврдено е дека пилот бил Дилокрит Патави, водачот на ескадрилата.'],
 ['Локалните медиуми известуваат за аеродромско противпожарно возило кое се превртело додека пожарникарите во него одговарале на повик.'],
 ['28-год

In [59]:
bleu.compute(predictions=predictions_new, references=references)

{'bleu': 0.1354328483702309,
 'precisions': [0.4327079934747145,
  0.1836734693877551,
  0.09458259325044405,
  0.05157992565055762],
 'brevity_penalty': 0.9651408402912618,
 'length_ratio': 0.9657345411579362,
 'translation_length': 2452,
 'reference_length': 2539}

In [60]:
chrf_score = sacrebleu.corpus_chrf(predictions_new, references)
print(f"chrF: {chrf_score.score:.2f}")

chrF: 55.36


## Few-Shot Cross-lingual prompt (ENG->MKD) in macedonian language

In [61]:
instruction = f"""Преведи го следниот текст од **Англиски** во **Македонски**. Само испечати го преводот на македонски јазик. Не додавај никакви објаснувања или дополнителни примери.
  Пример 1
  Англиски: On Monday, at 11 o'clock, a meeting has been scheduled between the Prime Minister of the United Kingdom and the Chancellor of Germany to discuss economic issues.
  Одговор (Македонски): Во понеделник, во 11 часот е договорена средба помеѓу премиерот на Велика Британија и канцеларот на Германија за разговори околу економските прашања.
  Пример 2
  Англиски: A sports match lasts 90 minutes, or two 45-minute halves with a 15-minute break in between. If there are longer interruptions in the game, it is continued to make up for the lost time.
  Одговор (Македонски): Еден спортски натпревар трае 90 минунти, односно две полувремиња по 45 минути со одмор меѓу нив во траење од 15 минути. Ако има подолги прекини во играта, таа се продолжува така што се надополнува изгубеното време.
  Пример 3
  Англиски: With the help of this tool, cells can be sorted by type, which means that diseases caused by small cells can be diagnosed.
  Одговор (Македонски): Со помош на оваа алатка, клетките можат да се подредуваат по тип, што значи дека можат да се дијагностицираат болести предизвикани од мали клетки.
  Пример 4
  Англиски: This study focuses on the development of new methods for data analysis.
  Одговор (Македонски): Оваа студија се фокусира на развојот на нови методи за анализа на податоци.
  Пример 5
  Англиски: In the 16th century BCE, the spear is believed to have been discovered as one of the most advanced weapons of that time.
  Одговор (Македонски): Во 16-ти век п.н.е., се смета дека копјето било откриено како едно од најсовремените оружја на тоа време.
"""

In [62]:
predictions_mkd = batch_translate(text_eng,instruction,max_new_tokens=128,lang_code="mkd",language="Македонски")

Batch 0
Во понеделник, научниците од Стanford Универзитет Сколова за медицина ја ануналисирале измишлуването на нова дијагностичка алатка која може да подредува клетките по тип: малиот печатлив чип кој може да се произведе со користење на стандардни инкжет принтери, и може да биде произведено за овозможување 

Batch 0
Истражувачки водачи вели дека ова може да доведе до рана детекција на рака, туберкулоза, HIV и маларија код пациенти во земји со нисок приход, каде што одживувањето од болести како што е ракот на дамското млечно засито може да биде половина од тоа на богати земји.

Текст:Lead researchers say this may bring 

Batch 0
Т-39C Грипен се натерао на трамвайна стаза во 9:30 часот локално време (0230 UTC) и се експлодира, што ја затвори аеродромот од комерцијални летови.

Текст: The JAS 39C Gripen crashed onto a runway at around 9:30 am local time (0230 UTC) and exploded, closing the airport to commercial flights.
Одговор (М 

Batch 0
Пилотот бил идентифициран како копајлер Дилокр

In [63]:
predictions_new = [first_sentence(text) for text in predictions_mkd]
predictions_new

['Во понеделник, научниците од Стanford Универзитет Сколова за медицина ја ануналисирале измишлуването на нова дијагностичка алатка која може да подредува клетките по тип: малиот печатлив чип кој може да се произведе со користење на стандардни инкжет принтери, и може да биде произведено за овозможување',
 'Истражувачки водачи вели дека ова може да доведе до рана детекција на рака, туберкулоза, HIV и маларија код пациенти во земји со нисок приход, каде што одживувањето од болести како што е ракот на дамското млечно засито може да биде половина од тоа на богати земји.',
 'Т-39C Грипен се натерао на трамвайна стаза во 9:30 часот локално време (0230 UTC) и се експлодира, што ја затвори аеродромот од комерцијални летови.',
 'Пилотот бил идентифициран како копајлер Дилокрит Патавеј.',
 'Локална медиа прикажува дека авионски возилото ја изгубило контрола и се наклонило на јужна страна.',
 '28-годишниот Видал се присојдени во Барца три сезони преди, од Севиља.',
 'Одкако се преселил во столичн

In [64]:
references = [[ref] for ref in text_mkd]
references[:10]

[['Во понеделникот научниците од медицинскиот факултет на Универзитетот Стенфорд објавија дека измислиле нова алатка за дијагностицирање со којашто се подредуваат клетките според типови: се работи за мал чип за печатење кој може да се произведе со помош на стандардни инк-џет печатачи, и тоа по цена од околу еден американски цент.'],
 ['Водечките истражувачи сметаат дека ова може да доведе до рано откривање на рак, туберколоза, ХИВ и маларија кај пациенти кои живеат во земји со низок животен стандард, каде стапката на преживување од болести како рак на дојка може да биде двојно пониска отколку во побогатите држави.'],
 ['JAS 39C Грипен се сруши на писта околу 9:30 am по локално време (2:30 UTC) и експлодираше, поради што аеродромот е затворен за комерцијални летови.'],
 ['Утврдено е дека пилот бил Дилокрит Патави, водачот на ескадрилата.'],
 ['Локалните медиуми известуваат за аеродромско противпожарно возило кое се превртело додека пожарникарите во него одговарале на повик.'],
 ['28-год

In [65]:
bleu.compute(predictions=predictions_new, references=references)

{'bleu': 0.13622642947092248,
 'precisions': [0.45427852348993286,
  0.19658493870402802,
  0.10027472527472528,
  0.04988009592326139],
 'brevity_penalty': 0.937051740900858,
 'length_ratio': 0.9389523434423002,
 'translation_length': 2384,
 'reference_length': 2539}

In [66]:
chrf_score = sacrebleu.corpus_chrf(predictions_new, references)
print(f"chrF: {chrf_score.score:.2f}")

chrF: 47.85


## Few-Shot Cross-lingual prompt (ENG->MKD) in serbian language

In [67]:
instruction = f"""Преведи следећи текст са **Eнглеског** на **Mакедонски** језик.
Само испиши превод на македонски језик. Не додавај никаква објашњења или додатне примере.

Пример 1
Енглески: On Monday, at 11 o'clock, a meeting has been scheduled between the Prime Minister of the United Kingdom and the Chancellor of Germany to discuss economic issues.
Одговор (Mакедонски): Во понеделник, во 11 часот е договорена средба помеѓу премиерот на Велика Британија и канцеларот на Германија за разговори околу економските прашања.

Пример 2
Енглески: A sports match lasts 90 minutes, or two 45-minute halves with a 15-minute break in between. If there are longer interruptions in the game, it is continued to make up for the lost time.
Одговор (Mакедонски): Еден спортски натпревар трае 90 минути, односно две полувремиња по 45 минути со одмор помеѓу нив во траење од 15 минути. Ако има подолги прекини во играта, таа се продолжува така што се надополнува изгубеното време.

Пример 3
Енглески: With the help of this tool, cells can be sorted by type, which means that diseases caused by small cells can be diagnosed.
Одговор (Mакедонски): Со помош на оваа алатка, клетките можат да се подредуваат по тип, што значи дека можат да се дијагностицираат болести предизвикани од мали клетки.

Пример 4
Енглески: This study focuses on the development of new methods for data analysis.
Одговор (Mакедонски): Оваа студија се фокусира на развојот на нови методи за анализа на податоци.

Пример 5
Енглески: In the 16th century BCE, the spear is believed to have been discovered as one of the most advanced weapons of that time.
Одговор (Mакедонски): Во 16-ти век п.н.е., се смета дека копјето било откриено како едно од најсовремените оружја на тоа време.
"""

In [68]:
predictions_mkd = batch_translate(text_eng,instruction,max_new_tokens=128,lang_code="srb",language="Македонски")

Batch 0
Во понеделник, научниците од Скола за медицина на Универзитетот Стanford ја објавија инвенцијата на нова дијагностичка алатка која може да подредува клетките по тип: малиот печатлив чип кој може да се произведе со користење на стандардни инкжет принтери за можно околу еден американски цент.
Само испиши превод на м 

Batch 0
Истражувачки водачи вели дека ова може да доведе до рана детекција на рака, туберкулоза, HIV и маларија код пациенти во земји со нисок приход, каде што ратите на преживување за болести како што е ракот на дамското млечно засито може да биде половина од онаа во богати земји.
Одговор (Макед 

Batch 0
JAS 39C Gripen-от се засекол на трамвайна стаза околу 9:30 часот локално време (0230 UTC) и се експлодирал, што ја затворило аеродромот од комерцијални летови.
The JAS 39C Gripen crashed onto a runway at around 9:30 am local time (0230 UTC) and exploded, closing the airport to commercial flights.
Одговор (Мак 

Batch 0
Пилотот бил идентифициран како копајлер Дилок

In [69]:
predictions_new = [first_sentence(text) for text in predictions_mkd]
predictions_new

['Во понеделник, научниците од Скола за медицина на Универзитетот Стanford ја објавија инвенцијата на нова дијагностичка алатка која може да подредува клетките по тип: малиот печатлив чип кој може да се произведе со користење на стандардни инкжет принтери за можно околу еден американски цент.',
 'Истражувачки водачи вели дека ова може да доведе до рана детекција на рака, туберкулоза, HIV и маларија код пациенти во земји со нисок приход, каде што ратите на преживување за болести како што е ракот на дамското млечно засито може да биде половина од онаа во богати земји.',
 'JAS 39C Gripen-от се засекол на трамвайна стаза околу 9:30 часот локално време (0230 UTC) и се експлодирал, што ја затворило аеродромот од комерцијални летови.',
 'Пилотот бил идентифициран како копајлер Дилокрит Патавеј.',
 'Локална медиа прикажува дека авионското возило за пожар е превртено додека одговара.',
 '28-годишниот Видал се присојдени во Барца три сезони преди, од Севиља.',
 'Одкако што се преселил во столичн

In [70]:
references = [[ref] for ref in text_mkd]
references[:10]

[['Во понеделникот научниците од медицинскиот факултет на Универзитетот Стенфорд објавија дека измислиле нова алатка за дијагностицирање со којашто се подредуваат клетките според типови: се работи за мал чип за печатење кој може да се произведе со помош на стандардни инк-џет печатачи, и тоа по цена од околу еден американски цент.'],
 ['Водечките истражувачи сметаат дека ова може да доведе до рано откривање на рак, туберколоза, ХИВ и маларија кај пациенти кои живеат во земји со низок животен стандард, каде стапката на преживување од болести како рак на дојка може да биде двојно пониска отколку во побогатите држави.'],
 ['JAS 39C Грипен се сруши на писта околу 9:30 am по локално време (2:30 UTC) и експлодираше, поради што аеродромот е затворен за комерцијални летови.'],
 ['Утврдено е дека пилот бил Дилокрит Патави, водачот на ескадрилата.'],
 ['Локалните медиуми известуваат за аеродромско противпожарно возило кое се превртело додека пожарникарите во него одговарале на повик.'],
 ['28-год

In [71]:
bleu.compute(predictions=predictions_new, references=references)

{'bleu': 0.1410835377626875,
 'precisions': [0.46523388116308473,
  0.2023757149142103,
  0.10308329498389324,
  0.05400192864030858],
 'brevity_penalty': 0.9324370418601744,
 'length_ratio': 0.9346199291059473,
 'translation_length': 2373,
 'reference_length': 2539}

In [72]:
chrf_score = sacrebleu.corpus_chrf(predictions_new, references)
print(f"chrF: {chrf_score.score:.2f}")

chrF: 55.76


## Few-Shot Cross-lingual prompt (ENG->MKD) in bulgarian language

In [73]:
instruction = f"""Преведи следния текст от **Aнглийски** на **Mакедонски** език.
Само отпечатай превода на македонски език. Не добавяй никакви обяснения или допълнителни примери.

Пример 1
Английски: On Monday, at 11 o'clock, a meeting has been scheduled between the Prime Minister of the United Kingdom and the Chancellor of Germany to discuss economic issues.
Отговор (Mакедонски): Во понеделник, во 11 часот е договорена средба помеѓу премиерот на Велика Британија и канцеларот на Германија за разговори околу економските прашања.

Пример 2
Английски: A sports match lasts 90 minutes, or two 45-minute halves with a 15-minute break in between. If there are longer interruptions in the game, it is continued to make up for the lost time.
Отговор (Mакедонски): Еден спортски натпревар трае 90 минути, односно две полувремиња по 45 минути со одмор помеѓу нив во траење од 15 минути. Ако има подолги прекини во играта, таа се продолжува така што се надополнува изгубеното време.

Пример 3
Английски: With the help of this tool, cells can be sorted by type, which means that diseases caused by small cells can be diagnosed.
Отговор (Mакедонски): Со помош на оваа алатка, клетките можат да се подредуваат по тип, што значи дека можат да се дијагностицираат болести предизвикани од мали клетки.

Пример 4
Английски: This study focuses on the development of new methods for data analysis.
Отговор (Mакедонски): Оваа студија се фокусира на развојот на нови методи за анализа на податоци.

Пример 5
Английски: In the 16th century BCE, the spear is believed to have been discovered as one of the most advanced weapons of that time.
Отговор (Mакедонски): Во 16-ти век п.н.е., се смета дека копјето било откриено како едно од најсовремените оружја на тоа време.
"""


In [74]:
predictions_mkd = batch_translate(text_eng,instruction,max_new_tokens=128,lang_code="bug",language="Македонски")

Batch 0
Во понеделник, научниците од Скола за медицина на Универзитетот Стanford ја објавија инвенцијата на нова дијагностичка алатка која може да подредува клетките по тип: малиот печатлив чип кој може да се произведе со користење на стандардни инкжет принтери, и тоа за можно околу еден американски цент.
Само отпечатай пр 

Batch 0
Истражувачки лидери вели дека ова може да доведе до рана детекција на рака, туберкулоза, HIV и маларија код пациенти во земји со нисок приход, каде што одзивот на болести како рак на млечните жлези може да биде половина од онаа во богати земји.
Само отпечатай превода на македонски език. Не добавяј 

Batch 0
The JAS 39C Gripen crashed onto a runway at around 9:30 am local time (0230 UTC) and exploded, closing the airport to commercial flights.

Текст: The JAS 39C Gripen crashed onto a runway at around 9:30 am local time (0230 UTC) and exploded, closing the airport to commercial flights.
Отговор (Македонски): The JAS 39C Gripen crashed onto a runway at around

In [75]:
predictions_new = [first_sentence(text) for text in predictions_mkd]
predictions_new

['Во понеделник, научниците од Скола за медицина на Универзитетот Стanford ја објавија инвенцијата на нова дијагностичка алатка која може да подредува клетките по тип: малиот печатлив чип кој може да се произведе со користење на стандардни инкжет принтери, и тоа за можно околу еден американски цент.',
 'Истражувачки лидери вели дека ова може да доведе до рана детекција на рака, туберкулоза, HIV и маларија код пациенти во земји со нисок приход, каде што одзивот на болести како рак на млечните жлези може да биде половина од онаа во богати земји.',
 'The JAS 39C Gripen crashed onto a runway at around 9:30 am local time (0230 UTC) and exploded, closing the airport to commercial flights.',
 'Пилотот бил идентифициран како копајлер Дилокрит Патавеј.',
 'Локална медиа извештава дека авионското возило за пожар е превртено додека одговарја.',
 '28-годишниот Видал се присојдил на Барца три сезони преди, од Севиља.',
 'Одкако се преселил во столична на Каталонија, Видал игра 49 натпревари за клуб

In [76]:
references = [[ref] for ref in text_mkd]
references[:10]

[['Во понеделникот научниците од медицинскиот факултет на Универзитетот Стенфорд објавија дека измислиле нова алатка за дијагностицирање со којашто се подредуваат клетките според типови: се работи за мал чип за печатење кој може да се произведе со помош на стандардни инк-џет печатачи, и тоа по цена од околу еден американски цент.'],
 ['Водечките истражувачи сметаат дека ова може да доведе до рано откривање на рак, туберколоза, ХИВ и маларија кај пациенти кои живеат во земји со низок животен стандард, каде стапката на преживување од болести како рак на дојка може да биде двојно пониска отколку во побогатите држави.'],
 ['JAS 39C Грипен се сруши на писта околу 9:30 am по локално време (2:30 UTC) и експлодираше, поради што аеродромот е затворен за комерцијални летови.'],
 ['Утврдено е дека пилот бил Дилокрит Патави, водачот на ескадрилата.'],
 ['Локалните медиуми известуваат за аеродромско противпожарно возило кое се превртело додека пожарникарите во него одговарале на повик.'],
 ['28-год

In [77]:
bleu.compute(predictions=predictions_new, references=references)

{'bleu': 0.152684578189533,
 'precisions': [0.4742138364779874,
  0.212253829321663,
  0.11304347826086956,
  0.06184084372003835],
 'brevity_penalty': 0.937470272842823,
 'length_ratio': 0.9393461992910594,
 'translation_length': 2385,
 'reference_length': 2539}

In [78]:
chrf_score = sacrebleu.corpus_chrf(predictions_new, references)
print(f"chrF: {chrf_score.score:.2f}")

chrF: 56.53
